# DX 704 Week 12 Project

This week's project will revisit the email spam classifier project from week 9 using large language model embeddings instead of custom features.


The full project description and a template notebook are available on GitHub: [Project 12 Materials](https://github.com/bu-cds-dx704/dx704-project-12).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [1]:
!wget https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

--2026-04-11 23:08:29--  https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip [following]
--2026-04-11 23:08:30--  https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15642124 (15M) [application/zip]
Saving to: ‘enron_spam_data.zip.2’

enron_spam_data.zip 100%[===================>]  14.92M  21.7MB/s    in 0.7s    

2026-04-11 23:08:30 (21.7 MB/s) - ‘enron_s

In [2]:
import pandas as pd

In [3]:
# pandas can read the zip file directly
enron_spam_data = pd.read_csv("enron_spam_data.zip")
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [4]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

np.float64(0.5092834262664611)

## Part 2: Download BERT Model

We will use a pre-trained BERT model to extract embedding vectors as described in lesson 2.1 this week.
Here is sample code to download the model from [Hugging Face](https://huggingface.co/) and extract one vector.
This model is small enough that you can run it with CPU only, but GPUs will be faster if available.

In [5]:
# You may need to install torch and transformers.
# Google Colab will have these installed already.
#
# pip install transformers torch --upgrade

import torch
from transformers import AutoTokenizer, AutoModel

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.to(device)
bert_model.eval()


I0000 00:00:1775963316.273566   74868 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [8]:
@torch.no_grad()
def embed_text(text):
    batch = [text]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
    outputs = bert_model(**inputs)
    # CLS token embedding is the first token's hidden state
    cls_emb = outputs.last_hidden_state[:, 0, :]  # shape: [batch_size, 768]
    return cls_emb.cpu()

In [9]:
v = embed_text("Hi, will you buy my spam?")
v.shape

torch.Size([1, 768])

## Part 3: Create Embedding Vectors

Use BERT to create embeddings for each email in the Enron data set.
You will have to decide how to combine the different columns of the original data set to produce one embedding vector.


Hint: BERT can be run without a GPU, but will be much slower.
Using Google Colab with only a CPU, it runs around 1 embedding per second.
Using Google Colab with the T4 GPU option, it runs around 60 embeddings per second.
Caching is also encouraged to avoid unnecessary reruns.

In [10]:
import json
import numpy as np
from pathlib import Path
from tqdm import tqdm

CACHE_PATH = Path("embeddings_cache.npy")
BATCH_SIZE = 64

def build_text(row):
    """Combine Subject and Message into one string for embedding."""
    subject = str(row["Subject"]) if pd.notna(row["Subject"]) else ""
    message = str(row["Message"]) if pd.notna(row["Message"]) else ""
    combined = (subject + " " + message).strip()
    return combined if combined else "[empty]"

@torch.no_grad()
def embed_batch(texts):
    """Embed a list of texts and return CLS token vectors as numpy array."""
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(device)
    outputs = bert_model(**inputs)
    cls_emb = outputs.last_hidden_state[:, 0, :]  # CLS token, shape: (batch, 768)
    return cls_emb.cpu().numpy()

# Load from cache if available to avoid re-running BERT on all 33k emails
if CACHE_PATH.exists():
    all_embeddings = np.load(CACHE_PATH)
    print(f"Loaded embeddings from cache: {all_embeddings.shape}")
else:
    texts = [build_text(row) for _, row in enron_spam_data.iterrows()]

    all_embeddings = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding emails"):
        batch_emb = embed_batch(texts[i : i + BATCH_SIZE])
        all_embeddings.append(batch_emb)

    all_embeddings = np.vstack(all_embeddings)
    np.save(CACHE_PATH, all_embeddings)
    print(f"Computed and cached embeddings: {all_embeddings.shape}")

print(f"Embedding matrix: {all_embeddings.shape}  (n_emails × 768)")

Embedding emails: 100%|██████████| 527/527 [10:17<00:00,  1.17s/it]

Computed and cached embeddings: (33716, 768)
Embedding matrix: (33716, 768)  (n_emails × 768)


Save your embeddings in a file "embeddings.tsv.gz" with two columns, Message ID and embedding_vector_json, where embedding_vector_json is a JSON-encoded list.
Make sure that embedding_vector_json is a 1 dimensional list, not 2 dimensional.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [11]:
embeddings_df = pd.DataFrame({
    "Message ID": enron_spam_data["Message ID"],
    "embedding_vector_json": [json.dumps(vec.tolist()) for vec in all_embeddings],
})

embeddings_df.to_csv("embeddings.tsv.gz", sep="\t", index=False)
print(f"Saved embeddings.tsv.gz — {len(embeddings_df):,} rows, 768-dim vectors")

Saved embeddings.tsv.gz — 33,716 rows, 768-dim vectors


Submit "embeddings.tsv.gz" in Gradescope.

## Part 4: Train a Linear Regression

Train an ordinary least squares regression for spam/ham status where spam is treated as target value 1 and ham is treated as target value 0 with your embeddings above as the only input variables.


In [12]:
from sklearn.linear_model import LinearRegression

# Labels: spam=1, ham=0
y = (enron_spam_data["Spam/Ham"] == "spam").astype(float).values
X = all_embeddings  # shape: (33716, 768)

lr_model = LinearRegression()
lr_model.fit(X, y)

# Sanity check: training accuracy at threshold 0.5
train_preds = (lr_model.predict(X) >= 0.5).astype(int)
train_acc = (train_preds == y.astype(int)).mean()
print(f"Training accuracy: {train_acc:.4f}")
print(f"Intercept: {lr_model.intercept_:.6f}")
print(f"Coefficients shape: {lr_model.coef_.shape}")

Training accuracy: 0.9811
Intercept: 0.926067
Coefficients shape: (768,)


Save the coefficients of your linear model in a file "coefficients.tsv" with columns dim and coefficient where dim specifies the offset in the embedding vector (0-767).
Don't worry about the bias term (but your model should still have it).

In [13]:
coef_df = pd.DataFrame({
    "dim": range(len(lr_model.coef_)),
    "coefficient": lr_model.coef_,
})

coef_df.to_csv("coefficients.tsv", sep="\t", index=False)
print(f"Saved coefficients.tsv — {len(coef_df)} rows (dim 0–767)")
print(coef_df.head())

Saved coefficients.tsv — 768 rows (dim 0–767)
   dim  coefficient
0    0     0.048700
1    1     0.007579
2    2     0.001954
3    3    -0.015280
4    4     0.034677


Submit "coefficients.tsv" in Gradescope.

## Part 5: Search for Relevant Documents

The file "queries.tsv" specifies 10 queries.
For each of the queries, encode them as a vector, and find the message that is closest using $L_2$.

In [14]:
from sklearn.neighbors import NearestNeighbors

queries = pd.read_csv("queries.tsv", sep="\t")
print(queries.to_string(index=False))

# Fit L2 nearest-neighbor index over all email embeddings
nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
nn.fit(all_embeddings)

# Embed each query (single texts — use embed_text from Part 2)
query_vectors = []
for _, row in queries.iterrows():
    vec = embed_text(row["query"]).numpy().squeeze()  # shape: (768,)
    query_vectors.append(vec)

query_vectors = np.array(query_vectors)  # shape: (10, 768)

# Find nearest email for each query
distances, indices = nn.kneighbors(query_vectors)

print("\nQuery → nearest email:")
for i, (_, row) in enumerate(queries.iterrows()):
    msg_id = enron_spam_data["Message ID"].iloc[indices[i, 0]]
    subject = enron_spam_data["Subject"].iloc[indices[i, 0]]
    print(f"  [{row['query_id']}] {row['query']!r:45s} → ID {msg_id}  ({subject!r:.60s})")

 query_id                                                query
        1                              accounting arrangements
        2                         sales higher than production
        3 asked lawyer to write letter about unexpected events
        4                                     engineering exam
        5                                   discounted tickets
        6                                 unexecuted agreement
        7                                            well bore
        8                                     capacity problem
        9                                       london partner
       10                                      dormant account

Query → nearest email:
  [1] 'accounting arrangements'                     → ID 3273  ('gymnastics pictures')
  [2] 'sales higher than production'                → ID 2663  ('first delivery atmic marquis')
  [3] 'asked lawyer to write letter about unexpected events' → ID 5057  ('rubbed his hands')
  [4] '

Save your results in a file "query-matches.tsv" with columns query_id, query_vector_json, and Message ID.

In [15]:
matches_df = pd.DataFrame({
    "query_id": queries["query_id"].values,
    "query_vector_json": [json.dumps(vec.tolist()) for vec in query_vectors],
    "Message ID": enron_spam_data["Message ID"].iloc[indices[:, 0]].values,
})

matches_df.to_csv("query-matches.tsv", sep="\t", index=False)
print(f"Saved query-matches.tsv — {len(matches_df)} rows")
print(matches_df[["query_id", "Message ID"]].to_string(index=False))

Saved query-matches.tsv — 10 rows
 query_id  Message ID
        1        3273
        2        2663
        3        5057
        4       13222
        5        3743
        6       21810
        7       18137
        8       14635
        9       14635
       10       15831


Submit "query-matches.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.